# Experiment 1 (Part 1 of 2): Tokenization Analysis with tiktoken and Hugging Face Tokenizers

**Course:** HAAI302COM - Artificial Intelligence Laboratory
**Experiment 1:** Tokenization Analysis and Text Generation using Hugging Face Transformers

## Aim
To explore tokenization strategies (Byte-Pair Encoding and WordPiece) using the `tiktoken` and Hugging Face `tokenizers` libraries, and to understand how text becomes the integer IDs that a language model actually reads.

## Objectives
1. Encode and decode text with OpenAI's `tiktoken` encodings and inspect every token.
2. Compare how GPT-2 (byte-level BPE), BERT (WordPiece) and Mistral (SentencePiece BPE) split the same words.
3. Measure token counts across English, Hindi, Marathi, code and numbers, and connect this to cost and context length.
4. Train a tiny BPE and a tiny WordPiece tokenizer from scratch and read the merge rules they learn.
5. Inspect padding, truncation, special tokens and attention masks for a batch.

Part 2 of this experiment (`02_Text_Generation_Decoding_Parameters.ipynb`) loads GPT-2 / Mistral and studies decoding parameters.

## Theory in one paragraph
A language model cannot read letters. It reads integers. A **tokenizer** is the dictionary that converts text into a list of integer IDs and back. Splitting by whole words gives a huge vocabulary and fails on new words; splitting by characters gives very long sequences. **Subword tokenizers** sit in between: frequent words stay whole, rare words are broken into reusable pieces. **BPE** learns its pieces by repeatedly merging the most frequent adjacent pair. **WordPiece** is similar but chooses the merge that most increases the likelihood of the training data, and marks word-continuation pieces with `##`.

## 0. Setup

We need four libraries:
- `tiktoken` - OpenAI's fast BPE tokenizer (used by GPT-3.5 / GPT-4 / GPT-4o).
- `transformers` - gives `AutoTokenizer`, which downloads the exact tokenizer that ships with a model on the Hugging Face Hub.
- `tokenizers` - the Rust library underneath `transformers`; we use it to train our own tokenizers.
- `pandas` - every intermediate result is shown as a DataFrame so you can read it like a table.

On Colab, uncomment the install line. The Mistral tokenizer is a small download (a few MB); the 7B model itself is not needed for this notebook.

In [1]:
# !pip install -q tiktoken transformers tokenizers pandas sentencepiece
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"      # keep download bars out of the saved outputs
import transformers, huggingface_hub
transformers.logging.set_verbosity_error()            # hide informational library warnings
huggingface_hub.utils.logging.set_verbosity_error()
import json
import pandas as pd
import tiktoken
from transformers import AutoTokenizer

pd.set_option("display.max_colwidth", 80)   # show long token strings without cutting them
print("tiktoken", tiktoken.__version__)

tiktoken 0.14.0


## 1. Why not just split on spaces or characters?

Before using any library, let us see the problem tokenizers solve. We take a tiny corpus and count how big the vocabulary would be at three levels:
- **character level** - every distinct character is a token,
- **word level** - every distinct space-separated word is a token,
- and we also measure how long a sentence becomes at each level.

What to watch: a small vocabulary means long sequences; a short sequence means a huge vocabulary.

In [2]:
corpus = [
    "The model reads tokens, not words.",
    "Tokenization converts text into integer IDs.",
    "Unbelievably, the tokenizer split unbelievable into pieces.",
    "Lower, lowest, newer, newest, wider, widest.",
    "Agentic AI systems call tools and read their outputs.",
]
chars = sorted(set("".join(corpus)))                       # distinct characters
words = sorted(set(w for line in corpus for w in line.split()))   # distinct words
sentence = corpus[2]
pd.DataFrame({
    "level": ["character", "word"],
    "vocabulary size": [len(chars), len(words)],
    "length of sentence 3": [len(sentence), len(sentence.split())],
})

,level,vocabulary size,length of sentence 3
0,character,32,59
1,word,33,7


**Takeaway.** Characters give a tiny vocabulary but turn one sentence into dozens of steps. Words give short sequences, but notice `pieces.` and `words.` are separate "words" because of the punctuation, and any word not seen during training (say `tokenizers`) would be unknown. Subword tokenization is the compromise every modern LLM uses.

## 2. tiktoken: the tokenizer behind GPT models

`tiktoken.get_encoding(name)` loads a ready-made BPE vocabulary. Three encodings matter historically:

| Encoding | Used by | Approx. vocabulary |
|---|---|---|
| `gpt2` | GPT-2, GPT-3 | 50k |
| `cl100k_base` | GPT-3.5, GPT-4 | 100k |
| `o200k_base` | GPT-4o and newer | 200k |

`enc.encode(text)` returns the list of IDs. `enc.n_vocab` is the vocabulary size.

In [3]:
encodings = {name: tiktoken.get_encoding(name) for name in ["gpt2", "cl100k_base", "o200k_base"]}
pd.DataFrame(
    [(name, enc.n_vocab) for name, enc in encodings.items()],
    columns=["encoding", "n_vocab"],
)

,encoding,n_vocab
0,gpt2,50257
1,cl100k_base,100277
2,o200k_base,200019


### 2.1 Look inside one encoding, token by token

`enc.encode()` gives integers, which tell us nothing. `enc.decode_single_token_bytes(id)` gives the raw bytes that one ID stands for, so we can line up ID and text side by side.

In [4]:
enc = encodings["cl100k_base"]
text = "Tokenization isn't magic: GPT-4 sees 12345 as pieces."
ids = enc.encode(text)
pieces = [enc.decode_single_token_bytes(i) for i in ids]   # bytes for each ID
pd.DataFrame({"token_id": ids, "bytes": pieces, "text": [p.decode("utf-8", "replace") for p in pieces]})

,token_id,bytes,text
0,3404,b'Token',Token
1,2065,b'ization',ization
2,4536,b' isn',isn
3,956,"b""'t""",'t
4,11204,b' magic',magic
5,25,b':',:
6,480,b' G',G
7,2898,b'PT',PT
8,12,b'-',-
9,19,b'4',4


**Takeaway.** Notice three things in the table:
1. Most tokens **start with a space** (`' sees'`, `' as'`). The space belongs to the following word, so `"sees"` and `" sees"` are different tokens.
2. `Tokenization` is split into common pieces, while short frequent words are a single token.
3. The number `12345` is split into chunks of up to three digits. This is one reason LLMs are weak at arithmetic: they never see a number as one unit.

### 2.2 Round trip: decode must give back the exact original text

A tokenizer is only useful if `decode(encode(text)) == text`. Byte-level BPE guarantees this for **any** input, including emojis or random bytes, because every byte (0-255) is in the base vocabulary.

In [5]:
samples = ["Hello world", "naive vs. naïve", "x = [i**2 for i in range(10)]", "नमस्ते"]
rows = []
for s in samples:
    ids = enc.encode(s)
    rows.append({"text": s, "n_tokens": len(ids), "round_trip_ok": enc.decode(ids) == s})
pd.DataFrame(rows)

,text,n_tokens,round_trip_ok
0,Hello world,2,True
1,naive vs. naïve,6,True
2,x = [i**2 for i in range(10)],13,True
3,नमस्ते,6,True


**Takeaway.** Every row round-trips perfectly, even the Hindi word `नमस्ते`. Byte-level BPE never produces an "unknown" token; at worst it falls back to individual bytes.

## 3. Token counts across languages, code and numbers

Token count directly controls **API cost** (you pay per token) and **how much fits in the context window**. Let us count tokens for the same kind of content in different languages and formats, across all three encodings.

In [6]:
texts = {
    "English": "Artificial intelligence is changing how we learn.",
    "Hindi":   "कृत्रिम बुद्धिमत्ता हमारे सीखने के तरीके को बदल रही है।",
    "Marathi": "कृत्रिम बुद्धिमत्ता आपण शिकण्याची पद्धत बदलत आहे.",
    "Python":  "def add(a, b):\n    return a + b",
    "Numbers": "3.14159 2718281828 1000000",
}
table = pd.DataFrame({name: {k: len(e.encode(v)) for k, v in texts.items()} for name, e in encodings.items()})
table["characters"] = [len(v) for v in texts.values()]
table

,gpt2,cl100k_base,o200k_base,characters
English,9,9,8,49
Hindi,90,55,17,55
Marathi,83,53,18,49
Python,15,11,11,31
Numbers,11,13,13,26


**Takeaway.**
- Hindi and Marathi sentences cost **several times more tokens** than an English sentence with the same meaning, especially with the old `gpt2` encoding. The model has to spend more of its context window and you pay more.
- Each newer encoding (bigger vocabulary) closes the gap, because it was trained on more multilingual text and learned whole Devanagari pieces.
- Code with indentation became cheaper in `cl100k_base`, which learned runs of spaces as single tokens.

### 3.1 Parameter experiment: vocabulary size vs tokens per character

A useful single number is **characters per token**. Higher means the tokenizer compresses that kind of text better.

In [7]:
ratio = table.drop(columns="characters").rdiv(table["characters"], axis=0).round(2)   # characters / tokens
ratio.columns = [f"chars/token ({c})" for c in ratio.columns]
ratio

,chars/token (gpt2),chars/token (cl100k_base),chars/token (o200k_base)
English,5.44,5.44,6.12
Hindi,0.61,1.00,3.24
Marathi,0.59,0.92,2.72
Python,2.07,2.82,2.82
Numbers,2.36,2.00,2.00


**Takeaway.** For English, a token is roughly 4 to 5 characters (the famous "one token is about three quarters of a word" rule). For Devanagari with the `gpt2` encoding it falls **below** 1 character per token (about 0.6), because each Devanagari character is 3 UTF-8 bytes and the tokenizer is working almost byte by byte. `o200k_base` lifts it to about 3.

## 4. Hugging Face tokenizers: three families side by side

`AutoTokenizer.from_pretrained(model_name)` downloads the tokenizer that was used to train that model. We load three:

| Model | Algorithm | How it marks word boundaries |
|---|---|---|
| `gpt2` | byte-level BPE | `Ġ` = "this token starts with a space" |
| `bert-base-uncased` | WordPiece | `##` = "this token continues the previous word" |
| `mistralai/Mistral-7B-v0.1` | SentencePiece BPE | `▁` = "this token starts with a space" |

`tokenizer.tokenize(text)` returns the string pieces; `tokenizer(text)["input_ids"]` returns the IDs.

In [8]:
toks = {
    "GPT-2 (BPE)":           AutoTokenizer.from_pretrained("gpt2"),
    "BERT (WordPiece)":      AutoTokenizer.from_pretrained("bert-base-uncased"),
    "Mistral (SentencePiece)": AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1"),
}
pd.DataFrame([(n, t.__class__.__name__, len(t)) for n, t in toks.items()], columns=["tokenizer", "class", "vocab size"])

,tokenizer,class,vocab size
0,GPT-2 (BPE),GPT2Tokenizer,50257
1,BERT (WordPiece),BertTokenizer,30522
2,Mistral (SentencePiece),TokenizersBackend,32000


In [9]:
word_tests = ["unbelievably", "tokenization", "ChatGPT", "Pune", "pneumonoultramicroscopic"]
rows = {name: [" | ".join(t.tokenize(w)) for w in word_tests] for name, t in toks.items()}
pd.DataFrame(rows, index=word_tests)

,GPT-2 (BPE),BERT (WordPiece),Mistral (SentencePiece)
unbelievably,un | bel | iev | ably,un | ##bel | ##ie | ##va | ##bly,▁un | bel | iev | ably
tokenization,token | ization,token | ##ization,▁token | ization
ChatGPT,Chat | G | PT,chat | ##gp | ##t,▁Chat | G | PT
Pune,P | une,pune,▁P | une
pneumonoultramicroscopic,p | neum | on | oult | ram | icro | sc | opic,p | ##ne | ##um | ##ono | ##ult | ##ram | ##ic | ##ros | ##copic,▁p | neum | on | oul | tr | am | icro | sc | opic


**Takeaway.**
- **BERT** lowercased `ChatGPT` before splitting (the model is *uncased*), and continuation pieces carry `##`.
- **GPT-2** keeps case, so `ChatGPT` and `chatgpt` would be completely different token sequences.
- **Mistral** puts `▁` on the first piece because SentencePiece treats the start of text as if a space were there.
- All three handle the made-up long medical word without an unknown token, by falling back to smaller pieces.

### 4.1 The leading-space trap (GPT-2)

In byte-level BPE the space is part of the token. So the *same word* gets *different IDs* depending on whether a space precedes it. This matters when you build prompts by string concatenation.

In [10]:
gpt2_tok = toks["GPT-2 (BPE)"]
variants = ["hello", " hello", "Hello", " Hello", "HELLO"]
pd.DataFrame({
    "text": [repr(v) for v in variants],
    "tokens": [gpt2_tok.tokenize(v) for v in variants],
    "ids": [gpt2_tok.encode(v) for v in variants],
})

,text,tokens,ids
0,'hello',[hello],[31373]
1,' hello',[Ġhello],[23748]
2,'Hello',[Hello],[15496]
3,' Hello',[ĠHello],[18435]
4,'HELLO',"[HE, LL, O]","[13909, 3069, 46]"


**Takeaway.** Five spellings of one word give five different ID sequences. The model learns each one separately, which is why a stray space or a capital letter in a prompt can change the output.

### 4.2 Looking at the learned merge rules

A BPE tokenizer is fully described by (a) its vocabulary and (b) its ordered list of **merges**. The merge list is stored inside the backend tokenizer's JSON. The first merges are the most frequent pairs in GPT-2's training data.

In [11]:
gpt2_state = json.loads(gpt2_tok.backend_tokenizer.to_str())    # the full tokenizer as JSON
merges = gpt2_state["model"]["merges"]
merges = [m if isinstance(m, str) else " ".join(m) for m in merges]   # newer versions store pairs as lists
print("number of merges:", len(merges))
pd.DataFrame({"rank": range(1, 11), "merge": merges[:10]})

number of merges: 50000


,rank,merge
0,1,Ġ t
1,2,Ġ a
2,3,h e
3,4,i n
4,5,r e
5,6,o n
6,7,Ġt he
7,8,e r
8,9,Ġ s
9,10,a t


**Takeaway.** The very first merges join `Ġ` with `t`, `a`, and so on, and form `he`, `in`, `the`. These are the most common byte pairs in English web text. GPT-2 has 50,000 merges on top of 256 base bytes plus one special token, which gives exactly 50,257 tokens.

### 4.3 The WordPiece vocabulary (BERT)

WordPiece has no merge list; it stores only the vocabulary and splits a word greedily, always taking the **longest** piece that exists in the vocabulary. `tokenizer.vocab` is a dict from string to ID. Let us count how many entries are continuation pieces (`##...`).

In [12]:
bert_tok = toks["BERT (WordPiece)"]
vocab = pd.Series(bert_tok.vocab).sort_values()              # token -> id, sorted by id
summary = pd.DataFrame({
    "kind": ["bracketed: [CLS], [SEP], [unused0]...", "continuation ##..", "word-start pieces"],
    "count": [vocab.index.str.match(r"^\[.*\]$").sum(), vocab.index.str.startswith("##").sum(),
              (~vocab.index.str.startswith("##") & ~vocab.index.str.match(r"^\[.*\]$")).sum()],
})
summary

,kind,count
0,"bracketed: [CLS], [SEP], [unused0]...",999
1,continuation ##..,5828
2,word-start pieces,23695


In [13]:
# Greedy longest-match in action: watch how BERT eats a long word from left to right
word = "unaffordability"
print(bert_tok.tokenize(word))
print("is '##ability' in vocab?", "##ability" in bert_tok.vocab, "| is '##abilities' in vocab?", "##abilities" in bert_tok.vocab)

['una', '##ff', '##ord', '##ability']
is '##ability' in vocab? True | is '##abilities' in vocab? False


**Takeaway.** Around a fifth of BERT's 30,522 entries are `##` continuation pieces. The 999 bracketed entries are the 5 real special tokens plus 994 `[unusedN]` placeholder slots that let you add your own tokens later without resizing the model. The split you see is the result of repeatedly grabbing the longest prefix that exists in the vocabulary.

## 5. Train your own BPE and WordPiece tokenizers from scratch

Reading a pre-trained tokenizer is useful, but the clearest way to understand BPE is to train one on a tiny corpus and look at the merges it learns. We use the `tokenizers` library:

- `models.BPE(...)` / `models.WordPiece(...)` - the algorithm.
- `pre_tokenizers.Whitespace()` - first split on spaces and punctuation, so merges never cross word boundaries.
- `trainers.BpeTrainer(vocab_size=..., min_frequency=...)`:
  - `vocab_size` - stop merging once the vocabulary reaches this size. **Small** = few merges, words stay in many pieces. **Large** = many merges, frequent words become single tokens.
  - `min_frequency` - a pair must appear at least this many times to be merged.

In [14]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

train_corpus = (["low lower lowest"] * 5 + ["new newer newest"] * 6 + ["wide wider widest"] * 3
                + ["slow slower slowest"] * 2 + ["the lowest price is the newest price"] * 2)
print(len(train_corpus), "training lines; sample:", train_corpus[:3])

18 training lines; sample: ['low lower lowest', 'low lower lowest', 'low lower lowest']


In [15]:
def train_bpe(vocab_size, min_frequency=2):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = pre_tokenizers.Whitespace()            # split on spaces first
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=min_frequency, special_tokens=["[UNK]"], show_progress=False)
    tok.train_from_iterator(train_corpus, trainer)              # learn merges from our lines
    return tok

bpe = train_bpe(vocab_size=40)
print("vocabulary size:", bpe.get_vocab_size())

vocabulary size: 40


### 5.1 Read the merges in the order they were learned

The first merge is the most frequent pair in the corpus. Each later merge is the most frequent pair *after* applying all previous merges.

In [16]:
learned = json.loads(bpe.to_str())["model"]["merges"]
learned = [m if isinstance(m, str) else " ".join(m) for m in learned]
pd.DataFrame({"step": range(1, len(learned) + 1), "merge (left right)": learned}).head(15)

,step,merge (left right)
0,1,w e
1,2,l o
2,3,n e
3,4,s t
4,5,we st
5,6,we r
6,7,d e
7,8,i de
8,9,w ide
9,10,lo west


**Takeaway.** The trainer starts from single characters and first merges the pairs that dominate the corpus: `w e`, `l o`, `n e`, `s t`, then `we st`. A few steps later whole words appear (`lo west` = `lowest`, `ne west` = `newest`). The order of the merges is the order of frequency in *this* corpus, which is why `-west` is built before `low` itself.

### 5.2 Parameter experiment: `vocab_size` too small vs too large

Train three BPE tokenizers with different `vocab_size` and tokenize the same words, including `slowly`, which never appears in the corpus.

In [17]:
test_words = "lowest newest widest slowly"
rows = {}
for vs in [20, 35, 60]:
    t = train_bpe(vocab_size=vs)
    rows[f"vocab_size={vs} (actual {t.get_vocab_size()})"] = t.encode(test_words).tokens
pd.DataFrame(dict((k, pd.Series(v)) for k, v in rows.items())).fillna("")

,vocab_size=20 (actual 20),vocab_size=35 (actual 35),vocab_size=60 (actual 41)
0,lo,lowest,lowest
1,west,newest,newest
2,ne,wide,widest
3,west,st,slow
4,w,s,l
5,i,low,[UNK]
6,d,l,
7,e,[UNK],
8,st,,
9,s,,


**Takeaway.**
- **Too small (20):** almost no merges were allowed, so words come out as single letters. Sequences become long and slow to process.
- **Middle (35):** the most frequent words (`lowest`, `newest`) are already single tokens, while the rarer `widest` is still split into `wide` + `st`.
- **Large (60):** frequent full words become one token. The trainer stops early (actual size below 60) once no pair meets `min_frequency`.
- The unseen word `slowly` is built from known pieces (`slow`, `l`) **except** the letter `y`, which never occurs in the corpus and becomes `[UNK]`. A character-level BPE can only reuse characters it has seen. GPT-2 avoids this with **byte-level** BPE: all 256 bytes are in the base vocabulary, so nothing is ever unknown (Section 2.2).

### 5.3 Train a WordPiece tokenizer on the same corpus

`models.WordPiece` uses `continuing_subword_prefix="##"` by default. We also attach a WordPiece **decoder** so that `decode()` glues `##` pieces back together.

In [18]:
from tokenizers import decoders

wp = Tokenizer(models.WordPiece(unk_token="[UNK]"))
wp.pre_tokenizer = pre_tokenizers.Whitespace()
wp.decoder = decoders.WordPiece()                     # join '##' pieces when decoding
wp.train_from_iterator(train_corpus, trainers.WordPieceTrainer(vocab_size=40, special_tokens=["[UNK]"], show_progress=False))
pd.DataFrame({"BPE (vocab 40)": pd.Series(bpe.encode(test_words).tokens),
              "WordPiece (vocab 40)": pd.Series(wp.encode(test_words).tokens)}).fillna("")

,BPE (vocab 40),WordPiece (vocab 40)
0,lowest,lowest
1,newest,newest
2,widest,wide
3,slow,##st
4,l,[UNK]
5,[UNK],


In [19]:
enc_wp = wp.encode("slowest widest")
print("tokens :", enc_wp.tokens)
print("ids    :", enc_wp.ids)
print("decoded:", wp.decode(enc_wp.ids))

tokens : ['slo', '##west', 'wide', '##st']
ids    : [39, 29, 33, 27]
decoded: slowest widest


**Takeaway.** Same corpus, same vocabulary budget, different splits. BPE marks nothing (the `Whitespace` pre-tokenizer already removed the spaces), while WordPiece marks every non-initial piece with `##` so the decoder knows where words end. The decoded text comes back as normal words.

## 6. From text to model input: special tokens, padding, truncation, attention mask

A model receives a **batch** of sequences, which must all be the same length. The tokenizer handles this:
- `padding=True` - pad shorter sequences up to the longest one in the batch; `padding="max_length"` pads everything to exactly `max_length`.
- `truncation=True, max_length=N` - cut longer sequences down to N tokens.
- `attention_mask` - 1 for real tokens, 0 for padding, so the model ignores the padding.
- `add_special_tokens` - BERT wraps every input as `[CLS] ... [SEP]`.

In [20]:
batch = ["AI agents use tools.", "Retrieval augmented generation grounds every answer in trusted source documents."]
out = bert_tok(batch, padding="max_length", truncation=True, max_length=10)
for i in range(len(batch)):
    display(pd.DataFrame({"token": bert_tok.convert_ids_to_tokens(out["input_ids"][i]),
                          "input_id": out["input_ids"][i], "attention_mask": out["attention_mask"][i]}).T)

,0,1,2,3,4,5,6,7,8,9
token,[CLS],ai,agents,use,tools,.,[SEP],[PAD],[PAD],[PAD]
input_id,101,9932,6074,2224,5906,1012,102,0,0,0
attention_mask,1,1,1,1,1,1,1,0,0,0


,0,1,2,3,4,5,6,7,8,9
token,[CLS],retrieval,augmented,generation,grounds,every,answer,in,trusted,[SEP]
input_id,101,26384,19335,4245,5286,2296,3437,1999,9480,102
attention_mask,1,1,1,1,1,1,1,1,1,1


**Takeaway.**
- Both rows have exactly 10 columns: the short sentence was **padded** with `[PAD]` (mask 0), the long one was **truncated** and still ends in `[SEP]`.
- `[CLS]` (id 101) and `[SEP]` (id 102) were added automatically.

### 6.1 GPT-2 has no pad token

Decoder-only models such as GPT-2 were trained on continuous text and ship without a pad token, so `padding=True` raises an error. The standard fix is to reuse the end-of-sequence token as padding and pad on the **left** for generation (so the last real token sits right next to where new tokens will be generated).

In [21]:
print("pad token before:", gpt2_tok.pad_token)
gpt2_tok.pad_token = gpt2_tok.eos_token       # reuse <|endoftext|> as padding
gpt2_tok.padding_side = "left"                # important for generation
out = gpt2_tok(["Hi", "Agentic AI is"], padding=True)
pd.DataFrame({"input_ids": out["input_ids"], "attention_mask": out["attention_mask"]}, index=["'Hi'", "'Agentic AI is'"])

pad token before: None


,input_ids,attention_mask
'Hi',"[50256, 50256, 50256, 17250]","[0, 0, 0, 1]"
'Agentic AI is',"[36772, 291, 9552, 318]","[1, 1, 1, 1]"


**Takeaway.** `'Hi'` was padded on the left with id 50256 (`<|endoftext|>`), and its attention mask is 0 there. You will reuse this exact setup in Part 2 when we generate text in batches.

## 7. Results summary

Fill in this table from your outputs above:

| Observation | Your value |
|---|---|
| Tokens for the English sentence (`cl100k_base`) | |
| Tokens for the Hindi sentence (`gpt2` vs `o200k_base`) | |
| BERT split of `unbelievably` | |
| GPT-2 split of `unbelievably` | |
| First merge learned by your BPE tokenizer | |

## Conclusion
Subword tokenizers balance vocabulary size against sequence length. BPE builds its vocabulary bottom-up by merging frequent pairs; WordPiece chooses merges by likelihood and marks continuation pieces with `##`. The choice of tokenizer affects cost, context usage, multilingual fairness and even arithmetic ability.

## Viva questions
1. Why does GPT-2 never produce an `[UNK]` token while a word-level tokenizer would?
2. What does the `Ġ` symbol in GPT-2 tokens mean, and what is its equivalent in Mistral?
3. Why does the same Hindi sentence cost more tokens than its English translation? Which encoding reduces this?
4. What happens to tokenization if `vocab_size` is set very small during training? Very large?
5. Why must decoder-only models be padded on the left during batched generation?

## Exercise
Take one paragraph from your own notes, count its tokens with all three `tiktoken` encodings, and estimate the cost of sending it 10,000 times at a price of your choice per million tokens.